# RegRAG: run every file, step by step

This notebook runs each file of the `regrag` package in order:

1. `config.py`: settings
2. `ingest.py`: PDFs → clause chunks
3. `index.py`: search indexes
4. `query.py`: understand the question
5. `retrieve.py`: find the best clauses
6. `amendments.py`: check for newer orders
7. `generate.py`: write the cited answer
8. `pipeline.py`: all of the above in one call
9. `graph.py`: the LangGraph agent

**Before running:**
- Start Ollama (`ollama serve`); it runs the embeddings.
- Have `GROQ_API_KEY` in `.env`; answers come from Groq `gpt-oss-120b`.
- Select the kernel **Python (RegRAG .venv)**.

Cells marked ⏱ call the LLM (a few seconds each on Groq). To run fully offline, set `LLM_PROVIDER=ollama` in `.env`.

## Setup

In [1]:
import logging
import os
import sys
import warnings

# Hide library warnings and progress bars so the outputs stay readable
warnings.filterwarnings("ignore")
logging.getLogger("pypdf").setLevel(logging.CRITICAL)   # pypdf logs harmless font-encoding messages for some PDFs
os.environ["TQDM_DISABLE"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# Go to the project folder so that "import regrag" works
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

print("Project folder:", os.getcwd())

Project folder: C:\Users\risha\OneDrive\Desktop\Mini Project


## 1. `config.py`: settings

All settings live in one file: folders, model names and search sizes.

In [2]:
from regrag import config

print("PDF folder:      ", config.RAW_DIR)
print("Answer model:    ", config.ANSWER_MODEL, f"({config.LLM_PROVIDER})")
print("Embedding model: ", config.EMBED_MODEL)
print("Reranker model:  ", config.RERANK_MODEL)
print("Results kept:    ", config.RERANK_TOP_N)
print("States covered:  ", config.STATES)

PDF folder:       C:\Users\risha\OneDrive\Desktop\Mini Project\data\raw
Answer model:     openai/gpt-oss-120b (groq)
Embedding model:  nomic-embed-text
Reranker model:   cross-encoder/ms-marco-MiniLM-L-6-v2
Results kept:     5
States covered:   ['Gujarat', 'Maharashtra', 'Karnataka', 'Tamil Nadu', 'Rajasthan']


## 2. `ingest.py`: PDFs → clause chunks

Reads every PDF listed in `data/manifest.json`, splits it at clause numbers (6, 6.2, 7.3…),
and saves the chunks to `data/processed/chunks.jsonl`.

In [3]:
from regrag import ingest

chunks = ingest.run()

GJ_GERC_NetMetering_Regulations_2016.pdf           48 chunks,  20 distinct clauses


GJ_GERC_NetMetering_Consolidated.pdf               47 chunks,  16 distinct clauses
GJ_GERC_NetMetering_4thAmendment_2024.pdf          10 chunks,   4 distinct clauses


MH_MERC_Rooftop_RE_Regulations_2019.pdf            61 chunks,  42 distinct clauses


MH_MERC_Rooftop_1stAmendment_SOR_2023.pdf          49 chunks,  13 distinct clauses


MH_MERC_Rooftop_2ndAmendment_SOR_2024.pdf          37 chunks,  13 distinct clauses


KA_KERC_Solar_TariffOrder_FY24.pdf                 37 chunks,  11 distinct clauses
SKIP KA_KERC_Order_2023.pdf: Scanned PDF with no extractable text; needs OCR (Tesseract) to include.
TN_TANGEDCO_PoliciesAdopted.pdf                     2 chunks,   0 distinct clauses


TN_TNERC_GISS_Regulations_2024_DRAFT.pdf          120 chunks, 100 distinct clauses


RJ_RERC_DRE_Regulations_2021.pdf                   89 chunks,  69 distinct clauses


RJ_RERC_SuoMotu_Order_2024.pdf                     14 chunks,  13 distinct clauses


IN_MoP_GreenEnergyOpenAccess_Rules_2022.pdf        13 chunks,   9 distinct clauses
Wrote 527 chunks -> C:\Users\risha\OneDrive\Desktop\Mini Project\data\processed\chunks.jsonl


In [4]:
# Look at one chunk
chunk = chunks[100]

print("Document:", chunk.doc_id)
print("State:   ", chunk.state)
print("Clause:  ", chunk.clause)
print("Date:    ", chunk.date)
print()
print(chunk.text[:400])

Document: GJ_NM_AMD4_2024
State:    Gujarat
Clause:   4
Date:     2024-09-04

4. Regulation 7 of the Principal Regulations shall be substituted as
under:
Various activities and different authorities are associated with the Solar
Rooftop PV project. It is necessary that the different entities carry out the
works within prescribed time limit. Therefore, time frame prescribed in the
table below shall be scrupulously followed by the concerned authorities.
Sr.
No.
Activity Sub A


## 3. `index.py`: search indexes

Two indexes over the same chunks:
- **ChromaDB** (meaning-based search) is saved in `data/chroma/`. Building it takes about 15 min on CPU,
  so we load the saved one. Set `BUILD_INDEX = True` to rebuild it.
- **BM25** (keyword search) is built in memory in about a second.

In [5]:
from regrag import index

BUILD_INDEX = False

if BUILD_INDEX or not config.CHROMA_DIR.exists():
    index.run()

store = index.load_vectorstore()
print("Chunks in ChromaDB:", store._collection.count())

Chunks in ChromaDB: 527


In [6]:
# Try the keyword (BM25) search on its own
bm25 = index.BM25Index(chunks)
results = bm25.search("net metering capacity limit", ["Rajasthan"], 3)

for chunk, score in results:
    print(round(score, 1), "|", chunk.doc_id, "| clause", chunk.clause)

9.0 | RJ_SUOMOTU_2024 | clause 10
7.1 | RJ_DREGS_2021 | clause 6.1
6.8 | RJ_SUOMOTU_2024 | clause 4


## 4. `query.py`: understand the question

Finds which state(s) a question is about, splits comparisons into one question per state,
and declines states we have no documents for.

In [7]:
from regrag import query

print(query.detect_states("What does BESCOM pay for rooftop solar?"))
print(query.detect_states("Net metering limit in Gujarat and Rajasthan?"))
print()

for sub in query.decompose("Compare transformer limits in Gujarat and Maharashtra"):
    print(sub.state, "->", sub.text)
print()

print(query.out_of_scope_message("What is the net metering limit in Kerala?"))

['Karnataka']
['Gujarat', 'Rajasthan']

Gujarat -> What are transformer limits in Gujarat?
Maharashtra -> What are transformer limits in Maharashtra?

I can't answer this: the indexed regulations cover only Gujarat, Maharashtra, Karnataka, Tamil Nadu, Rajasthan and central rules, not Kerala.


## 5. `retrieve.py`: find the best clauses

Runs BM25 and dense search (only inside the question's state), merges the two lists,
and reranks with a cross-encoder to keep the best 5. The first run downloads the reranker (~90 MB).

In [8]:
from regrag.retrieve import HybridRetriever

retriever = HybridRetriever()

question = "What is the current net metering capacity limit in Rajasthan?"
hits = retriever.retrieve(question, ["Rajasthan"])

for hit in hits:
    print(round(hit.rerank, 2), "|", hit.chunk.doc_id, "| clause", hit.chunk.clause, "|", hit.chunk.date)

5.94 | RJ_SUOMOTU_2024 | clause 10 | 2024-02-07
5.73 | RJ_SUOMOTU_2024 | clause 9 | 2024-02-07
5.07 | RJ_SUOMOTU_2024 | clause 6 | 2024-02-07
4.88 | RJ_SUOMOTU_2024 | clause 12 | 2024-02-07
4.74 | RJ_SUOMOTU_2024 | clause 7 | 2024-02-07


## 6. `amendments.py`: check for newer orders

If a later order changed a retrieved clause, the old clause is flagged and moved down.
Example: Gujarat's 2016 rule (clause 6.2) was amended in 2017.

In [9]:
from regrag.amendments import AmendmentChecker

checker = AmendmentChecker(retriever)

gujarat_question = "Is a residential consumer in Gujarat limited to 50% of sanctioned load for rooftop solar?"
gujarat_hits = retriever.retrieve(gujarat_question, ["Gujarat"])
checked_hits = checker.check(gujarat_question, gujarat_hits)

for hit in checked_hits:
    print(hit.chunk.doc_id, "| clause", hit.chunk.clause, "|", hit.chunk.date)
    for flag in hit.flags:
        print("    !", flag[:110])

GJ_NM_CONSOL | clause 6.2 | 2017-10-06
GJ_NM_CONSOL | clause 15 | 2017-10-06
GJ_NM_2016 | clause 13.6 | 2016-06-18
GJ_NM_CONSOL | clause 2.1 | 2017-10-06
    ! AMENDMENT - amends clause 2.1 of 'GERC (Net Metering Rooftop Solar PV Grid Interactive Systems) Regulations, 2
GJ_NM_CONSOL | clause 2.1 | 2017-10-06
    ! AMENDMENT - amends clause 2.1 of 'GERC (Net Metering Rooftop Solar PV Grid Interactive Systems) Regulations, 2
GJ_NM_2016 | clause 6.2 | 2016-06-18
    ! POSSIBLY SUPERSEDED - clause 6.2 (2016-06-18) is amended/revisited by 'GERC Net Metering Regulations, 2016 - c
GJ_NM_2016 | clause 2.1 | 2016-06-18
    ! POSSIBLY SUPERSEDED - clause 2.1 (2016-06-18) is amended/revisited by 'GERC Net Metering Regulations, 2016 - c


## 7. `generate.py`: write the cited answer ⏱

The LLM answers only from the sources and cites them as [S1], [S2]…
If the sources don't answer the question, it declines.

In [10]:
from regrag.generate import generate

answer = generate(gujarat_question, checked_hits)

print("Declined:", answer.abstained)
print()
print(answer.text)
print()
for c in answer.citations:
    print(c["id"], "|", c["title"][:60], "| clause", c["clause"], "|", c["date"])

Declined: False

No. Clause 6.2 of the GERC Net Metering Regulations 2016 (consolidated with First Amendment, 2017) states that for Residential Consumers “the Rooftop Solar PV System capacity shall be irrespective of their sanctioned load/contract demand,” while the 50 % cap applies only to non-residential consumers [S1]. Thus a residential consumer may install up to the overall limits of ≥ 1 kW and ≤ 1 MW, not limited to 50 % of sanctioned load. (Earlier clauses in the same regulation impose a 50 % cap for all consumers during the initial two years, but the specific residential exemption in 6.2 supersedes that general rule.)

S1 | GERC Net Metering Regulations, 2016 - consolidated with Firs | clause 6.2 | 2017-10-06


## 8. `pipeline.py`: everything in one call ⏱

`answer()` runs steps 4–7 for you: understand → retrieve → check amendments → generate.

In [11]:
from regrag import pipeline

result = pipeline.answer("What is the current net metering capacity limit in Rajasthan?")

print(result.answer)
print()
for c in result.citations:
    print(c["id"], "|", c["title"][:60], "| clause", c["clause"], "|", c["date"])

1 MW (one megawatt) is the current net-metering capacity limit for all consumer categories in Rajasthan. The RERC Suo-Motu Order dated 2024-02-07 amended the limit from 500 kW to 1 MW under regulation 7 of the RERC DREGS Regulation 2021 [S1]. Clause 12 of the same order expressly states that net-metering shall be applicable for loads up to one megawatt (01 MW) or up to the sanctioned load/contract demand, whichever is lower [S4]. This supersedes the earlier 500 kW ceiling noted in Clause 6 of the 2021 order [S3].

S1 | RERC Suo-Motu Order on net metering capacity under DREGS Reg | clause 10 | 2024-02-07
S3 | RERC Suo-Motu Order on net metering capacity under DREGS Reg | clause 6 | 2024-02-07
S4 | RERC Suo-Motu Order on net metering capacity under DREGS Reg | clause 12 | 2024-02-07


In [12]:
# A question about a state we don't cover is declined instantly (no LLM call)
result = pipeline.answer("What is the net metering limit in Kerala?")
print(result.answer)

I can't answer this: the indexed regulations cover only Gujarat, Maharashtra, Karnataka, Tamil Nadu, Rajasthan and central rules, not Kerala.


## 9. `graph.py`: the LangGraph agent ⏱

Same steps as the pipeline, plus a self-check: it verifies the answer against the sources,
and if it isn't supported, rewrites the question and tries once more.
Set `RUN_AGENT = False` to skip it.

In [13]:
from regrag import graph

RUN_AGENT = True

if RUN_AGENT:
    result, trace = graph.run("What is the current net metering capacity limit in Rajasthan?")

    print("Steps the agent took:")
    for step in trace:
        print("  -", step)
    print()
    print(result.answer)

Steps the agent took:
  - classify: routed to [Rajasthan]
  - retrieve[Rajasthan]: 5 hits, best rerank 5.94
  - amendments[Rajasthan]: 0 superseded flags, 0 amendment chunks pulled in
  - generate: answers drafted
  - verify[Rajasthan]: grounded=True
  - format: done

1 MW (one megawatt) is the current net-metering capacity limit for all consumer categories in Rajasthan [Clause 12, RERC Suo-Motu Order on net metering capacity under DREGS Regulations 2021, dated 2024-02-07][S4]. The order raised the limit from the earlier 500 kW (as per Clause 6) to 1 MW, applying “up to one megawatt (01 MW) or up to the sanctioned load/contract demand, whichever is lower” [Clause 12][S4].


## 10. Tests

Runs the 11 automated tests in `tests/test_core.py`. They don't need the LLM.

In [14]:
# Quotes are needed because the folder name "Mini Project" contains a space
!"{sys.executable}" -m pytest tests -q

............                                                             [100%]
12 passed in 1.42s


## 11. The web app (`app.py`)

The app uses the same `pipeline.answer()` you ran above. Start it from a terminal in the project folder:

```bash
.venv\Scripts\streamlit.exe run app.py
```

Then open http://localhost:8501.